In [3]:
import sys
sys.path.insert(0, '..')

In [5]:
import numpy as np
import duckdb
from src.retrieval.retriever import (
    load_catalog, load_2a_artifacts,
    search_2a, search_regulation,
    _get_va_coordinates, VAD_LEXICON
)

In [6]:
catalog = load_catalog()
model_2a, index_2a, map_2a = load_2a_artifacts()

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [7]:
TEST_QUERIES = [
    # (query, expected_valence_range, expected_energy_range)
    ("I feel melancholy and tired",        (0.0, 0.35), (0.0, 0.50)),
    ("I feel anxious and restless",        (0.0, 0.35), (0.50, 1.0)),
    ("I feel happy and energized",         (0.65, 1.0), (0.55, 1.0)),
    ("I feel calm and peaceful",           (0.55, 1.0), (0.0, 0.45)),
    ("I feel nostalgic and bittersweet",   (0.15, 0.50), (0.0, 0.50)),
    ("I feel stressed and overwhelmed",    (0.0, 0.35), (0.55, 1.0)),
    ("I feel focused and driven",          (0.45, 0.80), (0.40, 0.80)),
    ("I feel low energy and unmotivated",  (0.30, 0.70), (0.0, 0.45)),
]

In [8]:
# Metric 1 — VA alignment score
def va_alignment(results, expected_v_range, expected_e_range):
    """
    Fraction of returned tracks whose valence AND energy 
    fall within the expected range for this query.
    Higher = better retrieval quality.
    """
    correct = 0
    for r in results:
        v_ok = expected_v_range[0] <= r['valence'] <= expected_v_range[1]
        e_ok = expected_e_range[0] <= r['energy'] <= expected_e_range[1]
        if v_ok and e_ok:
            correct += 1
    return correct / len(results)

In [9]:
# Metric 2 — Mean VA distance from implied target
def mean_va_distance(results, target_v, target_e):
    """
    Average Euclidean distance in VA space between
    the query's implied coordinates and returned tracks.
    Lower = better.
    """
    distances = []
    for r in results:
        d = np.sqrt((r['valence'] - target_v)**2 + (r['energy'] - target_e)**2)
        distances.append(d)
    return np.mean(distances)

In [10]:
# Metric 3 — Cross-query differentiation (Jaccard distance)
def jaccard_distance(results_a, results_b):
    """
    How different are the top-k results for two queries?
    1.0 = completely different, 0.0 = identical.
    """
    ids_a = set(r['track_id'] for r in results_a)
    ids_b = set(r['track_id'] for r in results_b)
    intersection = len(ids_a & ids_b)
    union = len(ids_a | ids_b)
    jaccard_similarity = intersection / union if union > 0 else 0
    return 1 - jaccard_similarity  # distance, not similarity

In [12]:
# Phase 2A evaluation
print("=" * 60)
print("PHASE 2A EVALUATION")
print("=" * 60)

alignment_scores = []
distance_scores = []

for query, v_range, e_range in TEST_QUERIES:
    results = search_2a(query, model_2a, index_2a, map_2a, catalog, k=5)
    target_v, target_e = _get_va_coordinates(
        query, model_2a, index_2a, map_2a, catalog
    )
    
    alignment = va_alignment(results, v_range, e_range)
    distance = mean_va_distance(results, target_v, target_e)
    
    alignment_scores.append(alignment)
    distance_scores.append(distance)
    
    print(f"\n'{query}'")
    print(f"  Target VA: ({target_v:.3f}, {target_e:.3f})")
    print(f"  VA alignment: {alignment:.2f} ({int(alignment*5)}/5 tracks in range)")
    print(f"  Mean VA distance: {distance:.3f}")

print(f"\n{'='*60}")
print(f"OVERALL Phase 2A:")
print(f"  Mean VA alignment:  {np.mean(alignment_scores):.3f}")
print(f"  Mean VA distance:   {np.mean(distance_scores):.3f}")

# Cross-query differentiation
print(f"\n{'='*60}")
print("CROSS-QUERY DIFFERENTIATION")
print("=" * 60)

similar_pairs = [
    ("I feel sad and tired", "I feel anxious and restless"),
    ("I feel happy", "I feel energized and excited"),
    ("I feel calm", "I feel peaceful and serene"),
]

for q1, q2 in similar_pairs:
    r1 = search_2a(q1, model_2a, index_2a, map_2a, catalog, k=5)
    r2 = search_2a(q2, model_2a, index_2a, map_2a, catalog, k=5)
    jd = jaccard_distance(r1, r2)
    print(f"\n'{q1}' vs")
    print(f"'{q2}'")
    print(f"  Jaccard distance: {jd:.3f} ({'good' if jd > 0.6 else 'poor'} differentiation)")

PHASE 2A EVALUATION

'I feel melancholy and tired'
  Target VA: (0.157, 0.288)
  VA alignment: 1.00 (5/5 tracks in range)
  Mean VA distance: 0.132

'I feel anxious and restless'
  Target VA: (0.250, 0.843)
  VA alignment: 1.00 (5/5 tracks in range)
  Mean VA distance: 0.076

'I feel happy and energized'
  Target VA: (1.000, 0.735)
  VA alignment: 0.60 (3/5 tracks in range)
  Mean VA distance: 0.357

'I feel calm and peaceful'
  Target VA: (0.871, 0.104)
  VA alignment: 1.00 (5/5 tracks in range)
  Mean VA distance: 0.204

'I feel nostalgic and bittersweet'
  Target VA: (0.464, 0.460)
  VA alignment: 0.00 (0/5 tracks in range)
  Mean VA distance: 0.372

'I feel stressed and overwhelmed'
  Target VA: (0.254, 0.710)
  VA alignment: 1.00 (5/5 tracks in range)
  Mean VA distance: 0.118

'I feel focused and driven'
  Target VA: (0.298, 0.823)
  VA alignment: 0.00 (0/5 tracks in range)
  Mean VA distance: 0.036

'I feel low energy and unmotivated'
  Target VA: (0.561, 0.381)
  VA alignment: 